# 10 — Target Encoding for High-Cardinality Categoricals

**Difficulty:** ⭐⭐⭐⭐ | **Estimated time:** 60 min

**Week 12 theme:** Airbnb-style listing price prediction

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why naive target encoding causes **data leakage** and why that inflates training metrics.
2. Implement **leave-one-out (LOO) target encoding** from scratch.
3. Implement **k-fold mean target encoding** from scratch.
4. Apply **smoothing** to handle rare categories gracefully.
5. Encode correctly at **inference time** (fit on train, apply to test — no leakage).


## The Analogy: Class Exam Averages

Imagine you want to predict whether a student will pass based on their study group's average score.

**Naive approach:** calculate the group average *including the student's own score*, then use that average as a feature to predict the student's outcome.  
The problem: the student's own score is baked into the feature that's supposed to predict their score. You've told the model the answer while asking the question.

**Leave-one-out approach:** compute the average for everyone *else* in the group, then use *that* to predict the current student.  
This is honest — the feature genuinely only contains information about the student's peers, not the student themselves.

**K-fold approach:** split the class into 5 groups. For students in group 1, compute the encoding using groups 2–5. Then for group 2 use groups 1, 3, 4, 5. And so on. Every student gets encoded using data they were never part of.

In both honest approaches the model cannot "cheat" during training — the leakage gap between training R² and validation R² collapses.


## Why Does This Matter for ML?

Real-world datasets are full of **high-cardinality categorical features**:

| Feature | Cardinality |
|---|---|
| Airbnb neighbourhood | 30–300 |
| US zip code | ~30,000 |
| Product ID | millions |
| User ID | millions |

**One-hot encoding** these features is impractical:
- 30 neighbourhoods → 30 extra columns (manageable)
- 30,000 zip codes → 30,000 extra columns (sparse, memory-hungry, kills linear model performance)
- Most values are zero — almost no information per column

**Target encoding** collapses any categorical feature into a *single* numeric column that directly reflects the target signal in each category. One column, regardless of cardinality. The challenge is doing it without leaking the target into training.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Synthetic Airbnb dataset ──────────────────────────────────────────────────
N = 2000  # total listings
N_NEIGHBOURHOODS = 30

# Assign a "true" price premium per neighbourhood (varies a lot)
neighbourhood_names = [f'hood_{i:02d}' for i in range(N_NEIGHBOURHOODS)]
neighbourhood_premiums = np.random.normal(loc=0, scale=0.5, size=N_NEIGHBOURHOODS)

# Each listing gets a random neighbourhood
neighbourhood_ids = np.random.randint(0, N_NEIGHBOURHOODS, size=N)
neighbourhoods = [neighbourhood_names[i] for i in neighbourhood_ids]

# Numeric features: rooms, bathrooms, distance_to_center, review_score
rooms = np.random.randint(1, 6, size=N).astype(float)
bathrooms = np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0], size=N)
distance_km = np.abs(np.random.normal(3, 2, size=N))  # km from centre
review_score = np.clip(np.random.normal(4.3, 0.4, size=N), 1, 5)

# Target: log_price = linear combination of features + neighbourhood effect + noise
log_price = (
    4.5
    + 0.2 * rooms
    + 0.15 * bathrooms
    - 0.05 * distance_km
    + 0.3 * review_score
    + neighbourhood_premiums[neighbourhood_ids]  # neighbourhood signal
    + np.random.normal(0, 0.3, size=N)           # irreducible noise
)

df = pd.DataFrame({
    'neighbourhood': neighbourhoods,
    'rooms': rooms,
    'bathrooms': bathrooms,
    'distance_km': distance_km,
    'review_score': review_score,
    'log_price': log_price,
})

print(f'Dataset shape: {df.shape}')
print(f'Unique neighbourhoods: {df.neighbourhood.nunique()}')
print(f'Target (log_price) range: {log_price.min():.2f} – {log_price.max():.2f}')
df.head()

## Why One-Hot Encoding Fails at Scale

One-hot encoding (OHE) is fine for features with a handful of categories (say, ≤ 10).  
But it breaks down fast:

- **Dimensionality explosion:** each category becomes a new column. 30 neighbourhoods → 30 columns added. 30,000 zip codes → 30,000 columns added.
- **Sparsity:** each listing activates exactly one column. 29,999 out of 30,000 columns are zero. Extremely wasteful.
- **Rare-category problem:** a neighbourhood with only 2 listings gives the model almost nothing to learn from, yet gets its own dedicated column.
- **Inference brittleness:** if a test listing has a new neighbourhood never seen in training, the OHE column doesn't exist.

Let's see this concretely with our 30-neighbourhood dataset — and then imagine scaling to thousands of categories.


In [ ]:
np.random.seed(42)

NUMERIC_COLS = ['rooms', 'bathrooms', 'distance_km', 'review_score']

# Train/val split — we'll reuse this split throughout the notebook
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

# ── Baseline: numeric features only ──────────────────────────────────────────
X_train_num = df_train[NUMERIC_COLS].values
X_val_num   = df_val[NUMERIC_COLS].values
y_train = df_train['log_price'].values
y_val   = df_val['log_price'].values

ridge_base = Ridge(alpha=1.0)
ridge_base.fit(X_train_num, y_train)
r2_base_train = r2_score(y_train, ridge_base.predict(X_train_num))
r2_base_val   = r2_score(y_val,   ridge_base.predict(X_val_num))

# ── One-hot encoding: 30 neighbourhoods ──────────────────────────────────────
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# Fit only on training set — correct practice
X_train_ohe = ohe.fit_transform(df_train[['neighbourhood']])
X_val_ohe   = ohe.transform(df_val[['neighbourhood']])

# Combine numeric + OHE
X_train_full = np.hstack([X_train_num, X_train_ohe])
X_val_full   = np.hstack([X_val_num,   X_val_ohe])

ridge_ohe = Ridge(alpha=1.0)
ridge_ohe.fit(X_train_full, y_train)
r2_ohe_train = r2_score(y_train, ridge_ohe.predict(X_train_full))
r2_ohe_val   = r2_score(y_val,   ridge_ohe.predict(X_val_full))

print(f'Feature matrix shapes:')
print(f'  Numeric only : {X_train_num.shape}  →  {df_train.shape[0]} rows × {X_train_num.shape[1]} cols')
print(f'  With 30 OHE  : {X_train_full.shape}  →  {X_train_full.shape[1]} cols (added {X_train_ohe.shape[1]})')
print(f'  With 300 OHE : would be {X_train_num.shape[1] + 300} cols')
print(f'  With 30k OHE : would be {X_train_num.shape[1] + 30000} cols')
print()
print(f'R² scores:')
print(f'  Numeric only  — train: {r2_base_train:.3f}  val: {r2_base_val:.3f}')
print(f'  Numeric + OHE — train: {r2_ohe_train:.3f}  val: {r2_ohe_val:.3f}')

## Naive Target Encoding: Simple But Leaky

The simplest idea: replace each category with the **mean target value** for that category.

```
neighbourhood_encoded[i] = mean(log_price | neighbourhood == hood_05)
```

This compresses 30 columns back to 1. And the number *means something* — it reflects the true price level of that neighbourhood.

**The catch:** when computing the encoding on the training set, each row's own target value is included in the mean that becomes its feature. The target is leaking into the feature.  
The model during training sees: "this listing is in hood_05, and the average price in hood_05 (including this very listing) is $X." It has essentially been told the answer.

This inflates training R² and creates a large **leakage gap** (train R² − val R²).


In [ ]:
np.random.seed(42)

def naive_target_encoding(train_df, val_df, cat_col, target_col):
    """Replace each category with mean target computed from training data.
    
    For the training set this causes leakage — the current row's target
    is included in the mean that becomes that row's feature.
    """
    # Compute per-category mean on training data only
    means = train_df.groupby(cat_col)[target_col].mean()
    global_mean = train_df[target_col].mean()  # fallback for unseen categories

    # Map to training and validation sets
    train_encoded = train_df[cat_col].map(means).fillna(global_mean).values
    val_encoded   = val_df[cat_col].map(means).fillna(global_mean).values
    return train_encoded, val_encoded

train_naive, val_naive = naive_target_encoding(
    df_train, df_val, 'neighbourhood', 'log_price'
)

# Build feature matrices: numeric + naive encoding (1 column)
X_train_naive = np.hstack([X_train_num, train_naive.reshape(-1, 1)])
X_val_naive   = np.hstack([X_val_num,   val_naive.reshape(-1, 1)])

ridge_naive = Ridge(alpha=1.0)
ridge_naive.fit(X_train_naive, y_train)
r2_naive_train = r2_score(y_train, ridge_naive.predict(X_train_naive))
r2_naive_val   = r2_score(y_val,   ridge_naive.predict(X_val_naive))

print('Naive Target Encoding results:')
print(f'  Train R²      : {r2_naive_train:.3f}')
print(f'  Val   R²      : {r2_naive_val:.3f}')
print(f'  Leakage gap   : {r2_naive_train - r2_naive_val:.3f}  ← should be large')
print()
print('Note: the encoding is computed on the FULL training set.')
print('Each row\'s own target is included in its category mean → leakage.')

## Leave-One-Out (LOO) Target Encoding

The fix is conceptually simple: when computing the mean for row *i*, **exclude row *i* itself**.

For row *i* in category *c*:

$$\text{LOO}(i) = \frac{\sum_{j \in c,\, j \neq i} y_j}{|c| - 1}$$

We can compute this efficiently without a loop:

1. Compute **group sum** and **group count** for each category.
2. Subtract the current row's target from the sum, subtract 1 from the count.
3. Divide.

**Smoothing** adds a regularisation term that pulls rare-category estimates toward the global mean:

$$\text{smoothed}(i) = \frac{(n_{-i}) \times \text{LOO}(i) + \alpha \times \bar{y}}{n_{-i} + \alpha}$$

where $n_{-i} = |c| - 1$ and $\alpha$ is the smoothing strength.  
- $\alpha = 0$: no smoothing (same as unsmoothed LOO).
- Large $\alpha$: always predict global mean (useless but stable for rare categories).
- Good $\alpha$: somewhere in between, found by cross-validation.


In [ ]:
np.random.seed(42)

def loo_target_encoding(train_df, val_df, cat_col, target_col, smoothing=1.0):
    """Leave-one-out target encoding with optional smoothing.
    
    Training rows: each row is encoded using all OTHER rows in its category.
    Validation rows: encoded using the full training category mean (no LOO needed
    because val rows were never seen during the encoding computation).
    """
    global_mean = train_df[target_col].mean()  # grand mean for smoothing anchor

    # ── Training set: LOO ────────────────────────────────────────────────────
    # Total sum and count per category
    group_sum   = train_df.groupby(cat_col)[target_col].transform('sum')
    group_count = train_df.groupby(cat_col)[target_col].transform('count')

    # Subtract current row to get leave-one-out statistics
    loo_sum   = group_sum - train_df[target_col].values
    loo_count = group_count - 1

    # Avoid division by zero for singleton categories (clip count to ≥ 1)
    loo_mean = loo_sum / loo_count.clip(lower=1)

    # Apply smoothing: pull toward global_mean for small loo_count
    train_encoded = (
        (loo_count * loo_mean + smoothing * global_mean)
        / (loo_count + smoothing)
    ).values

    # ── Validation set: full category mean from training ──────────────────────
    # No LOO needed — val rows were never part of the encoding computation
    cat_stats = train_df.groupby(cat_col)[target_col].agg(['sum', 'count'])
    cat_mean  = cat_stats['sum'] / cat_stats['count']
    cat_count = cat_stats['count']

    def encode_val_row(cat):
        if cat in cat_mean.index:
            n = cat_count[cat]
            m = cat_mean[cat]
            return (n * m + smoothing * global_mean) / (n + smoothing)
        return global_mean  # unseen category → global mean

    val_encoded = val_df[cat_col].map(encode_val_row).values
    return train_encoded, val_encoded


train_loo, val_loo = loo_target_encoding(
    df_train, df_val, 'neighbourhood', 'log_price', smoothing=1.0
)

X_train_loo = np.hstack([X_train_num, train_loo.reshape(-1, 1)])
X_val_loo   = np.hstack([X_val_num,   val_loo.reshape(-1, 1)])

ridge_loo = Ridge(alpha=1.0)
ridge_loo.fit(X_train_loo, y_train)
r2_loo_train = r2_score(y_train, ridge_loo.predict(X_train_loo))
r2_loo_val   = r2_score(y_val,   ridge_loo.predict(X_val_loo))

print('LOO Target Encoding (smoothing=1.0):')
print(f'  Train R²      : {r2_loo_train:.3f}')
print(f'  Val   R²      : {r2_loo_val:.3f}')
print(f'  Leakage gap   : {r2_loo_train - r2_loo_val:.3f}  ← should be much smaller')
print()
print(f'Compare to naive gap: {r2_naive_train - r2_naive_val:.3f}')

In [ ]:
np.random.seed(42)

def kfold_target_encoding(train_df, val_df, cat_col, target_col,
                          n_splits=5, smoothing=1.0, random_state=42):
    """K-fold mean target encoding.
    
    For each fold, encode the fold's samples using statistics from all OTHER folds.
    This prevents any row from seeing its own target during encoding.
    Validation set is encoded using the full training set statistics.
    """
    from sklearn.model_selection import KFold

    global_mean = train_df[target_col].mean()
    train_encoded = np.full(len(train_df), global_mean)  # will be overwritten

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold_train_idx, fold_val_idx in kf.split(train_df):
        # Use the training portion of this fold to compute category statistics
        fold_train = train_df.iloc[fold_train_idx]
        group_stats = fold_train.groupby(cat_col)[target_col].agg(['sum', 'count'])

        # Encode the held-out portion of this fold
        for idx in fold_val_idx:
            cat = train_df.iloc[idx][cat_col]
            if cat in group_stats.index:
                n = group_stats.loc[cat, 'count']
                m = group_stats.loc[cat, 'sum'] / n
                # Smooth toward global mean proportional to group size
                train_encoded[idx] = (n * m + smoothing * global_mean) / (n + smoothing)
            else:
                train_encoded[idx] = global_mean  # category not in this fold's train split

    # ── Validation set: full training statistics ──────────────────────────────
    cat_stats = train_df.groupby(cat_col)[target_col].agg(['sum', 'count'])
    cat_mean  = cat_stats['sum'] / cat_stats['count']
    cat_count = cat_stats['count']

    def encode_val_row(cat):
        if cat in cat_mean.index:
            n = cat_count[cat]
            m = cat_mean[cat]
            return (n * m + smoothing * global_mean) / (n + smoothing)
        return global_mean

    val_encoded = val_df[cat_col].map(encode_val_row).values
    return train_encoded, val_encoded


train_kfold, val_kfold = kfold_target_encoding(
    df_train, df_val, 'neighbourhood', 'log_price',
    n_splits=5, smoothing=1.0, random_state=42
)

X_train_kf = np.hstack([X_train_num, train_kfold.reshape(-1, 1)])
X_val_kf   = np.hstack([X_val_num,   val_kfold.reshape(-1, 1)])

ridge_kf = Ridge(alpha=1.0)
ridge_kf.fit(X_train_kf, y_train)
r2_kf_train = r2_score(y_train, ridge_kf.predict(X_train_kf))
r2_kf_val   = r2_score(y_val,   ridge_kf.predict(X_val_kf))

print('K-Fold Target Encoding (5 folds, smoothing=1.0):')
print(f'  Train R²    : {r2_kf_train:.3f}')
print(f'  Val   R²    : {r2_kf_val:.3f}')
print(f'  Leakage gap : {r2_kf_train - r2_kf_val:.3f}')

## Smoothing: Handling Rare Categories

Consider a neighbourhood with only **2 listings**. The category mean from those 2 listings is a very noisy estimate. The model will over-fit to these rare neighbourhoods.

**Smoothing** blends the category estimate with the global mean:

$$\text{smoothed} = \frac{n \times \hat{\mu}_c + \alpha \times \bar{y}}{n + \alpha}$$

where:
- $n$ = number of samples in the category
- $\hat{\mu}_c$ = category mean
- $\bar{y}$ = global mean
- $\alpha$ = smoothing strength (a hyperparameter)

Intuition:
- **$n \gg \alpha$:** category mean dominates (we trust our estimate)
- **$n \ll \alpha$:** global mean dominates (we don't trust a rare category)
- $\alpha = 0$: no smoothing — rare categories get full weight even from 1 sample
- Large $\alpha$: everything collapses to global mean — no category signal at all

There's an optimal $\alpha$ in between. Let's find it empirically.


In [ ]:
np.random.seed(42)

# Vary smoothing strength across a wide range
alphas = [0, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100, 500]
r2_train_list, r2_val_list, gap_list = [], [], []

for alpha in alphas:
    tr, vl = kfold_target_encoding(
        df_train, df_val, 'neighbourhood', 'log_price',
        n_splits=5, smoothing=alpha, random_state=42
    )
    X_tr = np.hstack([X_train_num, tr.reshape(-1, 1)])
    X_vl = np.hstack([X_val_num,   vl.reshape(-1, 1)])
    ridge = Ridge(alpha=1.0).fit(X_tr, y_train)
    r2_tr = r2_score(y_train, ridge.predict(X_tr))
    r2_vl = r2_score(y_val,   ridge.predict(X_vl))
    r2_train_list.append(r2_tr)
    r2_val_list.append(r2_vl)
    gap_list.append(r2_tr - r2_vl)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: Train vs Val R² across alpha values
axes[0].plot(alphas, r2_train_list, 'o-', label='Train R²', color='steelblue')
axes[0].plot(alphas, r2_val_list,   's--', label='Val R²',   color='tomato')
axes[0].set_xscale('symlog', linthresh=0.1)  # log scale to spread out small values
axes[0].set_xlabel('Smoothing α')
axes[0].set_ylabel('R²')
axes[0].set_title('R² vs Smoothing Strength')
axes[0].legend()

# Plot 2: Leakage gap vs alpha
axes[1].plot(alphas, gap_list, 'D-', color='purple')
axes[1].axhline(0, color='grey', linestyle=':')
axes[1].set_xscale('symlog', linthresh=0.1)
axes[1].set_xlabel('Smoothing α')
axes[1].set_ylabel('Train R² − Val R²  (leakage gap)')
axes[1].set_title('Leakage Gap vs Smoothing Strength')

plt.tight_layout()
plt.show()

best_alpha = alphas[np.argmax(r2_val_list)]  # choose alpha that maximises val R²
print(f'Best smoothing α by val R²: {best_alpha}')
print(f'α=0 (no smoothing) gap : {gap_list[0]:.4f}')
print(f'α=500 (heavy)      gap : {gap_list[-1]:.4f}')
print(f'Best α             gap : {gap_list[alphas.index(best_alpha)]:.4f}')

In [ ]:
# ── Optional: compare with category_encoders.TargetEncoder ───────────────────
np.random.seed(42)

try:
    from category_encoders import TargetEncoder

    te = TargetEncoder(cols=['neighbourhood'], smoothing=1.0)
    # fit_transform on train, transform on val
    df_train_ce = df_train.copy()
    df_val_ce   = df_val.copy()
    df_train_ce = te.fit_transform(df_train_ce[['neighbourhood']], y_train)
    df_val_ce   = te.transform(df_val[['neighbourhood']])

    X_tr_ce = np.hstack([X_train_num, df_train_ce.values])
    X_vl_ce = np.hstack([X_val_num,   df_val_ce.values])
    r_ce = Ridge(alpha=1.0).fit(X_tr_ce, y_train)
    print('category_encoders TargetEncoder available.')
    print(f'  Train R²: {r2_score(y_train, r_ce.predict(X_tr_ce)):.3f}')
    print(f'  Val   R²: {r2_score(y_val,   r_ce.predict(X_vl_ce)):.3f}')

except ImportError:
    print('category_encoders not installed — using scratch implementations (validated above).')
    print('Install with: pip install category_encoders')
    # Confirm scratch implementation is consistent
    print(f'\nScratch K-Fold TE val R²: {r2_kf_val:.3f}  (our trusted baseline)')

In [ ]:
np.random.seed(42)

# ── Correct workflow for test-time encoding ───────────────────────────────────
# Rule: encoding statistics are computed on TRAINING DATA ONLY.
# The test set is mapped using those statistics — it never influences them.

# 1. Compute category mean and count on the training set
SMOOTHING = 1.0
global_mean_train = df_train['log_price'].mean()

cat_stats_train = df_train.groupby('neighbourhood')['log_price'].agg(['sum', 'count'])
cat_mean_train  = cat_stats_train['sum'] / cat_stats_train['count']
cat_count_train = cat_stats_train['count']

# 2. Build a smoothed lookup table — one number per neighbourhood
smoothed_means = (
    (cat_count_train * cat_mean_train + SMOOTHING * global_mean_train)
    / (cat_count_train + SMOOTHING)
)

# 3. Apply to test (val) set — .map fills unseen categories with NaN; fillna with global mean
val_te_correct = (
    df_val['neighbourhood']
    .map(smoothed_means)
    .fillna(global_mean_train)
    .values
)

# 4. No leakage is possible — val targets were never involved
X_val_correct = np.hstack([X_val_num, val_te_correct.reshape(-1, 1)])
ridge_correct = Ridge(alpha=1.0).fit(
    np.hstack([X_train_num, smoothed_means[df_train['neighbourhood']].values.reshape(-1, 1)]),
    y_train
)
r2_correct_val = r2_score(y_val, ridge_correct.predict(X_val_correct))

print('Correct test-time encoding workflow:')
print(f'  Val R² (correct): {r2_correct_val:.3f}')
print()
print('Smoothed lookup table (first 5 neighbourhoods):')
print(smoothed_means.head())

In [ ]:
np.random.seed(42)

# ── Final comparison table ────────────────────────────────────────────────────
results = {
    'Method': [
        'Numeric Only',
        'Numeric + OHE (30 cols)',
        'Naive Target Encoding',
        'LOO Target Encoding',
        'K-Fold Target Encoding',
    ],
    'Train R²': [
        r2_base_train,
        r2_ohe_train,
        r2_naive_train,
        r2_loo_train,
        r2_kf_train,
    ],
    'Val R²': [
        r2_base_val,
        r2_ohe_val,
        r2_naive_val,
        r2_loo_val,
        r2_kf_val,
    ],
    'Leaky?': ['No', 'No', 'YES', 'No', 'No'],
    'Cols added': [0, 30, 1, 1, 1],
}

results_df = pd.DataFrame(results)
results_df['Leakage gap'] = (results_df['Train R²'] - results_df['Val R²']).round(3)
results_df['Train R²'] = results_df['Train R²'].round(3)
results_df['Val R²']   = results_df['Val R²'].round(3)

print('\n=== Final Comparison ===')
print(results_df.to_string(index=False))

# Visual: leakage gap bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue', 'steelblue', 'tomato', 'mediumseagreen', 'mediumseagreen']
ax.barh(results_df['Method'], results_df['Leakage gap'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Train R² − Val R²  (leakage gap)')
ax.set_title('Leakage gap by encoding method\n(smaller is better; red = leaky)')
plt.tight_layout()
plt.show()

## Self-Check Questions

**Q1.** You train a model with naive target encoding and observe that training R² = 0.92 but validation R² = 0.61. What is most likely causing this gap, and how would you fix it?

<details><summary>Show answer</summary>

The large gap (0.31) is almost certainly **data leakage from naive target encoding**. When computing category means on the training set, each row's own target value is included in the mean that becomes its feature. The model effectively "sees" the target during training, inflating R². During validation the model no longer has access to the current sample's target, so performance collapses.

**Fix:** Use leave-one-out or k-fold target encoding instead. Both exclude the current sample's target when computing the encoding for that sample.

</details>

---

**Q2.** A neighbourhood appears only once in your training data. Without smoothing, what value does LOO encoding assign to that row? Why is this problematic?

<details><summary>Show answer</summary>

With only 1 sample in the neighbourhood, LOO leaves **zero samples** to compute a mean from ($n - 1 = 0$). The LOO encoding is undefined (0/0), and our implementation clips to avoid division by zero, falling back to an arbitrary value.

This is problematic because the encoding provides no meaningful signal. With smoothing ($\alpha > 0$), the encoding falls back toward the **global mean**, which is a sensible default: we don't know anything specific about this neighbourhood, so assume average pricing.

</details>

---

**Q3.** At inference time you receive a listing from a neighbourhood that was never seen during training. What should your encoding return, and why?

<details><summary>Show answer</summary>

Return the **global mean** of the target from the training set. This is the best estimate available when we have zero category-specific information. Some implementations return the global mean directly; others use heavy smoothing to produce the same effect. Never use the test set's target to compute the encoding — that would be data leakage.

</details>


## Key Takeaways

1. **Never use naive target encoding in practice.** It causes severe data leakage that inflates training metrics and makes models appear better than they are.

2. **K-fold or LOO encoding are the correct approaches.** Both ensure that no row's target influences its own encoding. K-fold is generally preferred because the implementation is cleaner and matches the cross-validation workflow you're already using.

3. **Smooth for rare categories.** A category with 2 samples has a very noisy mean. Blending toward the global mean with smoothing factor $\alpha$ stabilises rare-category encodings.

4. **At inference time, always encode from training statistics.** Compute category means on the training set. Apply those fixed means to any new data. Never recompute on test data — that is leakage.

5. **Target encoding is powerful for high-cardinality categoricals.** It collapses thousands of categories into a single informative numeric column, which is far more practical than one-hot encoding for zip codes, product IDs, or user IDs.
